##Question Metadata
Domain: Omics based risk / disease prediction (GWAS, PRS, bulk, single‑cell
RNA‑seq, proteins / metabolite signatures associated with diseases)

Subdomain: GWAS summary statistics processing & loci identification

Topic tags: GWAS, genome-wide significance, lead SNP, locus clumping, summary stats

Programming Language: Python

Tools: pandas, numpy

# TASK PROMPT

##Distance-based Identification of Independent GWAS Loci from Genome-wide Significant Hits

## Problem Description

Given GWAS summary statistics, identify independent genome-wide significant loci by applying predefined filtering and distance-based locus-definition rules that approximate standard GWAS post-processing procedures. The task converts variant-level association results into reproducible locus-level discoveries using deterministic computational criteria, ensuring that nearby correlated variants are summarized as independent genetic signals.

## Background

Genome-wide association studies (GWAS) identify genetic variants associated with complex traits and diseases by testing millions of variants across the genome for statistical associations with a phenotype. Due to linkage disequilibrium, nearby variants are often inherited together, causing association signals to appear as clusters rather than isolated variants. Consequently, GWAS analyses summarize results at the locus level by grouping nearby significant variants and selecting a representative lead variant for each region. Defining independent loci prevents redundant reporting of correlated signals and enables consistent comparison across studies. Implementing this process using clearly defined computational rules promotes reproducibility and supports objective evaluation of analytical workflows.

## Task Requirements

Your analysis must identify independent genome-wide significant loci from GWAS summary statistics using deterministic filtering and distance-based locus definition rules. Perform the following steps:

1. **Input Parsing**: Load the GWAS summary statistics file containing at minimum the columns: SNP, Chromosome, Position, and P.

2. **Data Validation**: Exclude rows with missing or invalid values in Chromosome, Position, or P. Remove variants where P ≤ 0 or P > 1.

3. **Genome-wide Significance Filtering**: Retain only variants satisfying the genome-wide significance threshold: P < 5e-8

4. **Chromosome-wise Processing**:Process variants independently for each chromosome to ensure loci are defined within genomic context.

5. **Variant Ordering** : Sort variants by Chromosome (ascending) and Genomic position (ascending)

6. **Preliminary Chunk Definition** : Within each chromosome, assign variants into preliminary groups (“chunks”) such that a new chunk begins whenever the distance between consecutive variants is ≥ 500,000 base pairs.

7. **Chunk Lead Selection** : For each chunk, select a representative lead variant using deterministic ranking by choosing the variant with the smallest P-value; if multiple variants share the same P-value, select the one with the smallest genomic position, and if a tie still remains, select the variant with the lexicographically smallest SNP identifier.

8. **Locus Merging** : Merge adjacent chunk leads into a single locus when their genomic positions differ by < 1,000,000 base pairs.

9. **Locus Boundary Definition** : For each merged locus, compute genomic boundaries where the start position is defined as max(0, minimum position − 500,000) and the end position as maximum position + 500,000, and represent each locus using the format Chromosome:start_end.

10. **Independent Lead Variant Selection** : Within each locus, select exactly one independent lead variant using the deterministic ranking defined in Step 7.

11. **Output Construction** : Generate a structured output table containing:

locus_id
Chromosome
lead_SNP
lead_Position
lead_P
locus

12. **Output Ordering** : Sort loci by chromosome and lead position.
Assign sequential locus_id values starting from 1.

13. **Deterministic Requirement** : The procedure must produce identical outputs for identical inputs and must not use randomness.

## Implementation Instructions

Edit the solution/solution.py file to implement the required functionality. The input GWAS summary statistics files are preformatted and contain all required columns, so you can focus on implementing the filtering, locus-definition, and deterministic selection logic described in the task requirements.

### Function Signature

Your solution must implement the following function:

```python
def solution(file_path: str):
    """
    Identify independent genome-wide significant loci from GWAS summary statistics
    using deterministic filtering and distance-based locus-definition rules.

    Args:
        file_path (str): Path to a directory containing:
            - gwas_summary.csv: GWAS summary statistics file with columns:
                SNP (str), Chromosome (int), Position (int), P (float)
              (Additional columns may be present and must be ignored.)

    Returns:
        list: A list of dictionaries, one per independent locus, each with:
            {
                "locus_id": int,           # 1-indexed, assigned after sorting loci by chr then lead position
                "chromosome": int,         # Chromosome number
                "lead_snp": str,           # Selected lead variant identifier
                "lead_position": int,      # Genomic position (bp) of lead variant
                "lead_p": float,           # P value of lead variant
                "locus": str               # Locus boundary string: "{chromosome}:{start}_{end}"
            }

        The list must be sorted by:
            1) chromosome ascending
            2) lead_position ascending

    Notes:
        - Apply genome-wide significance filtering: P < 5e-8.
        - Define preliminary chunks per chromosome using a 500,000 bp gap rule.
        - Merge adjacent chunk leads into loci if lead positions differ by < 1,000,000 bp.
        - Locus boundaries: start = max(0, min_pos - 500,000), end = max_pos + 500,000.
        - Lead selection is deterministic: smallest P, then smallest Position, then smallest SNP lexicographically.
    
    """
    pass
```

## Input Format

The `file_path` parameter points to a directory containing:
- `gwas_summary.csv`: GWAS summary statistics file containing at minimum the following columns:

  - `SNP` (string): Variant identifier

  - `Chromosome` (integer): Chromosome number

  - `Position` (integer): Genomic position in base pairs

  - `P` (float): Association p-value

Additional columns may be present but must be ignored by the solution.

## Output Format

Return a list of dictionaries, one per independent locus, each containing:
| Key | Type | Description |
|-----|------|-------------|
| `locus_id` | int | Sequential integer locus ID |
| `chromosome` | int | Chromosome number |
| `lead_snp` | str | Lead variant identifier |
| `lead_position` | int | Genomic position (bp) |
| `lead_p` | float | P-value of lead variant |
| `locus` | str | Locus boundary string |

### Example Output

```python
[
    {
        "locus_id": 1,
        "chromosome": 1,
        "lead_snp": "rs1",
        "lead_position": 100000,
        "lead_p": 1e-9,
        "locus": "1:0_650000"
    },
    {
        "locus_id": 2,
        "chromosome": 1,
        "lead_snp": "rs3",
        "lead_position": 900000,
        "lead_p": 3e-9,
        "locus": "1:400000_1400000"
    },
    {
        "locus_id": 3,
        "chromosome": 2,
        "lead_snp": "rs5",
        "lead_position": 300000,
        "lead_p": 4e-9,
        "locus": "2:0_1350000"
    }
    # ... additional loci if present
]
```


## Example Data

Example GWAS summary statistics are provided in `example_data/`:

- `example_data/gwas_summary.csv`
This dataset is provided solely for illustration and must not be used for evaluation of hidden test cases.

## Hints

This task requires careful consideration of:

- Genomic Distance Relationships: Nearby variants may represent the same association signal; genomic distance rules must be applied consistently when grouping variants into loci.

- Deterministic Ranking: Lead variants must be selected using strict ordering rules (P-value, position, then SNP identifier) to ensure reproducible results.

- Chromosome-wise Processing: Variants from different chromosomes must be handled independently to avoid incorrect locus merging.

- Edge Case Handling: Special cases such as tied P-values, single-variant loci, or loci near genomic position 0 must be handled explicitly.

- Output Consistency: Results must follow the exact output schema and ordering requirements to enable automated validation.

#Expert Solution Code

In [1]:
import os
import pandas as pd
from typing import List, Dict, Tuple


SIG_THRESHOLD = 5e-8
GAP_WINDOW_BP = 500_000          # new chunk if consecutive gap >= 500kb
MERGE_LEAD_DIST_BP = 1_000_000   # merge adjacent chunk-leads if distance < 1Mb
BOUNDARY_PAD_BP = 500_000        # locus boundaries extend +/- 500kb


def _coerce_numeric(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce Chromosome, Position, P to numeric and drop invalid rows."""
    df = df.copy()
    df["Chromosome"] = pd.to_numeric(df["Chromosome"], errors="coerce")
    df["Position"] = pd.to_numeric(df["Position"], errors="coerce")
    df["P"] = pd.to_numeric(df["P"], errors="coerce")

    df = df.dropna(subset=["SNP", "Chromosome", "Position", "P"])
    df["Chromosome"] = df["Chromosome"].astype(int)
    df["Position"] = df["Position"].astype(int)

    # Valid p-values only
    df = df[(df["P"] > 0) & (df["P"] <= 1)]
    return df


def _deterministic_pick_lead(group: pd.DataFrame) -> pd.Series:
    """
    Deterministic lead selection:
      1) smallest P
      2) smallest Position
      3) lexicographically smallest SNP
    """
    g = group.sort_values(by=["P", "Position", "SNP"], ascending=[True, True, True])
    return g.iloc[0]


def _assign_chunks_with_gap_rule(df_chr: pd.DataFrame) -> pd.DataFrame:
    """
    Assign preliminary chunks within a chromosome based on the 500kb gap rule:
    start new chunk when consecutive position gap >= 500,000 bp.
    """
    d = df_chr.sort_values(["Position", "P", "SNP"]).reset_index(drop=True).copy()
    chunk_ids = []
    current_chunk = 1

    prev_pos = None
    for pos in d["Position"].tolist():
        if prev_pos is None:
            chunk_ids.append(current_chunk)
        else:
            if (pos - prev_pos) >= GAP_WINDOW_BP:
                current_chunk += 1
            chunk_ids.append(current_chunk)
        prev_pos = pos

    d["chunk"] = chunk_ids
    return d


def _merge_chunk_leads_into_locus_groups(leads_chr: pd.DataFrame) -> pd.DataFrame:
    """
    Merge adjacent chunk leads into locus groups if lead positions differ by < 1,000,000 bp.
    Returns leads with an additional 'locus_group' integer id.
    """
    d = leads_chr.sort_values(["Position", "P", "SNP"]).reset_index(drop=True).copy()
    groups = []
    current_group = 1
    prev_pos = None

    for pos in d["Position"].tolist():
        if prev_pos is None:
            groups.append(current_group)
        else:
            if (pos - prev_pos) < MERGE_LEAD_DIST_BP:
                groups.append(current_group)
            else:
                current_group += 1
                groups.append(current_group)
        prev_pos = pos

    d["locus_group"] = groups
    return d


def solution(file_path: str) -> List[Dict]:
    """
    Identify independent genome-wide significant loci from GWAS summary statistics
    using deterministic filtering and distance-based locus-definition rules.

    Expects file_path directory to contain:
      - gwas_summary.csv with columns SNP, Chromosome, Position, P
    """
    in_file = os.path.join(file_path, "gwas_summary.csv")
    if not os.path.exists(in_file):
        raise FileNotFoundError(f"Expected input file not found: {in_file}")

    # Load (allow extra columns, ignore them)
    df = pd.read_csv(in_file)

    required = {"SNP", "Chromosome", "Position", "P"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    df = df[list(required)].copy()
    df = _coerce_numeric(df)

    # Genome-wide significance filtering
    df = df[df["P"] < SIG_THRESHOLD].copy()
    if df.empty:
        return []

    # Sort for stable processing
    df = df.sort_values(["Chromosome", "Position", "P", "SNP"]).reset_index(drop=True)

    loci_records: List[Dict] = []

    for chr_num in sorted(df["Chromosome"].unique()):
        df_chr = df[df["Chromosome"] == chr_num].copy()
        if df_chr.empty:
            continue

        # Step 1: preliminary chunking by 500kb gap rule
        df_chr = _assign_chunks_with_gap_rule(df_chr)

        # Step 2: pick one deterministic lead per chunk
        leads = (
            df_chr.groupby("chunk", as_index=False)
            .apply(lambda g: _deterministic_pick_lead(g))
        )
        # groupby.apply returns a DataFrame with weird index; normalize
        if isinstance(leads, pd.Series):
            leads = leads.to_frame().T
        leads = leads.reset_index(drop=True)

        # Step 3: merge chunk leads into locus groups if lead distance < 1Mb
        leads = _merge_chunk_leads_into_locus_groups(leads)

        # Step 4: compute locus boundaries from *all variants* belonging to merged groups
        # Map each original chunk to its locus_group via the leads table
        chunk_to_group = dict(zip(leads["chunk"].astype(int), leads["locus_group"].astype(int)))
        df_chr["locus_group"] = df_chr["chunk"].map(chunk_to_group).astype(int)

        # For each locus_group, compute boundaries and select final deterministic lead SNP
        for lg in sorted(df_chr["locus_group"].unique()):
            grp = df_chr[df_chr["locus_group"] == lg].copy()

            start = max(0, int(grp["Position"].min()) - BOUNDARY_PAD_BP)
            end = int(grp["Position"].max()) + BOUNDARY_PAD_BP
            locus_str = f"{chr_num}:{start}_{end}"

            lead_row = _deterministic_pick_lead(grp)

            loci_records.append(
                {
                    # locus_id assigned later after sorting
                    "chromosome": int(chr_num),
                    "lead_snp": str(lead_row["SNP"]),
                    "lead_position": int(lead_row["Position"]),
                    "lead_p": float(lead_row["P"]),
                    "locus": locus_str,
                }
            )

    # Final ordering: chromosome asc, then lead_position asc
    loci_records.sort(key=lambda x: (x["chromosome"], x["lead_position"], x["lead_p"], x["lead_snp"]))

    # Assign locus_id sequentially
    for i, rec in enumerate(loci_records, start=1):
        rec["locus_id"] = i

    return loci_records

#Test Cases

In [ ]:
"""
Test suite for Identification of Independent Genome-Wide Significant Loci
"""

# ==================== BASIC STRUCTURE & FORMAT ====================

def test_output_structure():
    """Test that solution returns correct data structure."""
    file_path = "/.test_data"
    result = solution(file_path)

    # Top-level structure
    assert isinstance(result, list), "Result must be a list"

    # Each locus dict must have required keys
    required_keys = {"locus_id", "chromosome", "lead_snp", "lead_position", "lead_p", "locus"}
    for locus in result:
        assert isinstance(locus, dict), "Each locus must be a dictionary"
        assert set(locus.keys()) == required_keys, f"Must have exactly these keys: {required_keys}"


def test_field_types_and_ranges():
    """Test that each field has valid type/range."""
    file_path = "/.test_data"
    result = solution(file_path)

    for r in result:
        assert isinstance(r["locus_id"], int), "locus_id must be int"
        assert isinstance(r["chromosome"], int), "chromosome must be int"
        assert isinstance(r["lead_snp"], str), "lead_snp must be str"
        assert isinstance(r["lead_position"], int), "lead_position must be int"
        assert isinstance(r["lead_p"], float), "lead_p must be float"
        assert isinstance(r["locus"], str), "locus must be str"

        assert r["chromosome"] > 0, "chromosome must be > 0"
        assert r["lead_position"] >= 0, "lead_position must be >= 0"
        assert 0.0 < r["lead_p"] <= 1.0, "lead_p must be in (0, 1]"


# ==================== CORE RULES ====================

def test_significance_threshold():
    """All lead variants must be genome-wide significant."""
    file_path = "/.test_data"
    result = solution(file_path)

    # Empty list is allowed if no variants pass the threshold in a hidden test set
    for r in result:
        assert r["lead_p"] < 5e-8, "lead_p must be < 5e-8"


def test_locus_string_format():
    """Locus string must follow 'chr:start_end' and have valid integer boundaries."""
    file_path = "/.test_data"
    result = solution(file_path)

    for r in result:
        locus = r["locus"]
        chr_part, coords = locus.split(":")
        start_str, end_str = coords.split("_")

        assert chr_part.isdigit(), "Chromosome in locus must be numeric"
        assert start_str.isdigit(), "Start must be a non-negative integer"
        assert end_str.isdigit(), "End must be a non-negative integer"

        start = int(start_str)
        end = int(end_str)
        assert end >= start, "End must be >= start"


def test_ordering_and_locus_ids():
    """Check sorting and sequential locus_id assignment."""
    file_path = "/.test_data"
    result = solution(file_path)

    # locus_id sequential
    ids = [r["locus_id"] for r in result]
    assert ids == list(range(1, len(result) + 1)), "locus_id must be sequential starting at 1"

    # sorted by chromosome then lead_position
    pairs = [(r["chromosome"], r["lead_position"]) for r in result]
    assert pairs == sorted(pairs), "Results must be sorted by chromosome, then lead_position"


def test_determinism():
    """Same input must produce identical output."""
    file_path = "/.test_data"
    r1 = solution(file_path)
    r2 = solution(file_path)
    assert r1 == r2, "Solution must be deterministic"


# ==================== RUN ALL TESTS ====================

test_output_structure()
test_field_types_and_ranges()
test_significance_threshold()
test_locus_string_format()
test_ordering_and_locus_ids()
test_determinism()

print("All tests passed!")